In [25]:
import os

In [26]:
server_url = os.environ.get("SERVER_URL")
api_key = os.environ.get("MISTRAL_API_KEY")

In [27]:
from mistralai import Mistral  # noqa: E402
from mistralai.models import UserMessage, SystemMessage  # noqa: E402

In [28]:
client = Mistral(
    server_url=server_url,
    api_key=api_key
)

In [29]:
import pydantic
class InvoiceData(pydantic.BaseModel):
    """Structured invoice data extracted by LLM."""
    invoice_number: str
    date: str
    total_amount: float
    bank_details: str
    supplier: str
    invoice_category: str



In [31]:
ocr_result = """
Invoice no: 23285582

Date of issue: 12/19/2015

**Seller:**

Harris-Green
97868 Adam Parkways
Salazarshire, WI 44155
Tax Id: 935-80-2834
IBAN: GB11QYTR88328261828600

**Client:**

Good PLC
55068 Dawn Mount Suite 471
Brookstown, KS 60292
Tax Id: 993-98-0454

**ITEMS**

|  No. | Description | Qty | UM | Net price | Net worth | VAT [%] | Gross worth  |
| --- | --- | --- | --- | --- | --- | --- | --- |
|  1. | 16 Inches Marble Coffee Table
Top Inlay with Multi Stone at
Border Game Table | 4,00 | each | 280,50 | 1 122,00 | 10% | 1 234,20  |
|  2. | 6'x3' Black Marble Large Dining
Table Top Very Fine Handmade
Marquetry Birds Art | 5,00 | each | 5 303,66 | 26 518,30 | 10% | 29 170,13  |
|  3. | Luxurious Pattern Stone Dinning
Table Top Marble Conference
Table Size 48 Inches | 2,00 | each | 3 492,00 | 6 984,00 | 10% | 7 682,40  |

**SUMMARY**

|   | VAT [%] | Net worth | VAT | Gross worth  |
| --- | --- | --- | --- | --- |
|   | 10% | 34 624,30 | 3 462,43 | 38 086,73  |
|  Total |  | $ 34 624,30 | $ 3 462,43 | $ 38 086,73  |
"""


In [ ]:
SYSTEM_PROMPT = """
You are an agent specialised in extracting invoice data and categorising invoices. You will be provided with invoices under markdown format, and will be tasked to extract relevant information and categorise the invoice.

Required fields:
- invoice_number: The invoice number
- date: The invoice date (format: YYYY-MM-DD)
- total_amount: The total amount of the invoice (numeric value)
- bank_details: The bank account details (RIB)
- supplier: The supplier who sent the bill
- invoice_category: The category of the invoice. One of: financial_expense (bank fees, etc.), subscriptions (phones, software licences, etc), furniture (office supplies), raw_materials (purchase of good for production), rent (office rentals), professional_services (accountants, lawyers, consultants), expenses (travel, meals), marketing (ads, content creation), or unknown

If any field cannot be determined, use "unknown" for strings or 0 for numeric values.
"""
USER_PROMPT = """
Extract the following information from the invoice text below:

Invoice text:
{raw_text}

###
Provide the extracted information in the required format.
"""
SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "date",
        "invoice_number",
        "total_amount",
        "supplier",
        "invoice_category",
        "bank_details",
    ],
    "properties": {
        "date": {"type": "string", "description": "Date of the invoice"},
        "supplier": {"type": "string", "description": "supplier"},
        "bank_details": {
            "type": "string",
            "description": "The bank details of the supplier",
        },
        "total_amount": {"type": "number", "description": ""},
        "invoice_number": {"type": "string", "description": ""},
        "invoice_category": {
            "type": "string",
            "enum": [
                "financial_expense",
                "subscriptions",
                "furniture",
                "raw_materials",
                "rent",
                "professional_services",
                "expenses",
                "marketing",
                "unknown",
            ],
            "description": "Category of the spend",
        },
    },
}


response = client.chat.complete(
    model="mistral-large-latest",
    messages=[
        SystemMessage(content=SYSTEM_PROMPT),
        UserMessage(content=USER_PROMPT.format(raw_text=ocr_result)),
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "invoice_data",
            "schema": SCHEMA,
            "strict": True,
        },
    },
)

raw_content = response.choices[0].message.content
if not isinstance(raw_content, str):
    raise ValueError(f"Unexpected content type: {type(raw_content)}")

invoice_data = InvoiceData.model_validate_json(raw_content)
print(invoice_data)


In [18]:
response

ChatCompletionResponse(id='b7d54cc3a3284a0ca444f8d3e932228e', object='chat.completion', model='mistral-medium-latest', usage=UsageInfo(prompt_tokens=36, completion_tokens=25, total_tokens=61, prompt_audio_seconds=Unset(), prompt_tokens_details={'cached_tokens': 0}), created=1773246548, choices=[ChatCompletionChoice(index=0, message=AssistantMessage(role='assistant', content='{\n  "extracted_information": {\n    "text": "fdvfd activit ###"\n  }\n}', tool_calls=None, prefix=False), finish_reason='stop')])

In [19]:
raw_content = response.choices[0].message.content
raw_content


'{\n  "extracted_information": {\n    "text": "fdvfd activit ###"\n  }\n}'

In [37]:
import os
from dotenv import load_dotenv
load_dotenv()

server_url = os.environ.get("SERVER_URL")
api_key = os.environ.get("MISTRAL_API_KEY")

from mistralai import Mistral
from mistralai.models import UserMessage, SystemMessage
import pydantic

client = Mistral(
    server_url=server_url,
    api_key=api_key
)

class InvoiceData(pydantic.BaseModel):
    invoice_number: str
    date: str
    total_amount: float
    bank_details: str
    supplier: str
    invoice_category: str

ocr_result = """
Invoice no: 23285582

Date of issue: 12/19/2015

**Seller:**

Harris-Green
97868 Adam Parkways
Salazarshire, WI 44155
Tax Id: 935-80-2834
IBAN: GB11QYTR88328261828600

**Client:**

Good PLC
55068 Dawn Mount Suite 471
Brookstown, KS 60292
Tax Id: 993-98-0454

**ITEMS**

|  No. | Description | Qty | UM | Net price | Net worth | VAT [%] | Gross worth  |
| --- | --- | --- | --- | --- | --- | --- | --- |
|  1. | 16 Inches Marble Coffee Table Top Inlay with Multi Stone at Border Game Table | 4,00 | each | 280,50 | 1 122,00 | 10% | 1 234,20  |
|  2. | 6'x3' Black Marble Large Dining Table Top Very Fine Handmade Marquetry Birds Art | 5,00 | each | 5 303,66 | 26 518,30 | 10% | 29 170,13  |
|  3. | Luxurious Pattern Stone Dinning Table Top Marble Conference Table Size 48 Inches | 2,00 | each | 3 492,00 | 6 984,00 | 10% | 7 682,40  |

**SUMMARY**

|   | VAT [%] | Net worth | VAT | Gross worth  |
| --- | --- | --- | --- | --- |
|   | 10% | 34 624,30 | 3 462,43 | 38 086,73  |
|  Total |  | $ 34 624,30 | $ 3 462,43 | $ 38 086,73  |
"""

print("Setup complete. server_url:", server_url, "| api_key set:", bool(api_key))


Setup complete. server_url: https://api.mistral.ai | api_key set: True


In [39]:
SYSTEM_PROMPT = """
You are an agent specialised in extracting invoice data and categorising invoices. You will be provided with invoices under markdown format, and will be tasked to extract relevant information and categorise the invoice.

Required fields:
- invoice_number: The invoice number
- date: The invoice date (format: YYYY-MM-DD)
- total_amount: The total amount of the invoice (numeric value)
- bank_details: The bank account details (RIB)
- supplier: The supplier who sent the bill
- invoice_category: The category of the invoice. One of: financial_expense (bank fees, etc.), subscriptions (phones, software licences, etc), furniture (office supplies), raw_materials (purchase of good for production), rent (office rentals), professional_services (accountants, lawyers, consultants), expenses (travel, meals), marketing (ads, content creation), or unknown

If any field cannot be determined, use "unknown" for strings or 0 for numeric values.
"""
USER_PROMPT = """
Extract the following information from the invoice text below:

Invoice text:
{raw_text}

###
Provide the extracted information in the required format.
"""
SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "date",
        "invoice_number",
        "total_amount",
        "supplier",
        "invoice_category",
        "bank_details",
    ],
    "properties": {
        "date": {"type": "string", "description": "Date of the invoice"},
        "supplier": {"type": "string", "description": "supplier"},
        "bank_details": {
            "type": "string",
            "description": "The bank details of the supplier",
        },
        "total_amount": {"type": "number", "description": ""},
        "invoice_number": {"type": "string", "description": ""},
        "invoice_category": {
            "type": "string",
            "enum": [
                "financial_expense",
                "subscriptions",
                "furniture",
                "raw_materials",
                "rent",
                "professional_services",
                "expenses",
                "marketing",
                "unknown",
            ],
            "description": "Category of the spend",
        },
    },
}

response = client.chat.complete(
    model="mistral-large-latest",
    messages=[
        SystemMessage(content=SYSTEM_PROMPT),
        UserMessage(content=USER_PROMPT.format(raw_text=ocr_result)),
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "invoice_data",
            "schema": SCHEMA,
            "strict": True,
        },
    },
)

raw_content = response.choices[0].message.content
if not isinstance(raw_content, str):
    raise ValueError(f"Unexpected content type: {type(raw_content)}")

invoice_data = InvoiceData.model_validate_json(raw_content)
print(invoice_data)


invoice_number='23285582' date='2015-12-19' total_amount=38086.73 bank_details='GB11QYTR88328261828600' supplier='Harris-Green' invoice_category='furniture'
